## OpenRouter full question bank test
Run all 42 questions from the question bank through a chosen OpenRouter model and save responses to JSONL format.
This notebook uses a default OpenRouter model ID and writes results to `data/responses/openrouter.jsonl` unless the override environment variable is changed.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import openai

cwd = Path.cwd()
dotenv_path = None
for candidate in [cwd / ".env.local", cwd.parent / ".env.local", cwd.parent.parent / ".env.local"]:
    if candidate.exists():
        dotenv_path = candidate
        break
if not dotenv_path:
    raise ValueError(".env.local not found. Place it in the repo root or export OPENROUTER_API_KEY manually.")
load_dotenv(dotenv_path)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found. Set it in .env.local or export it.")
client = openai.OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
)


print("Loaded OpenRouter client")

In [ ]:
import json
import time
from datetime import datetime, timezone
import pandas as pd

# Load the question bank
question_csv = Path("../figures/question_bank.csv")
questions_df = pd.read_csv(question_csv)

print(f"Loaded {len(questions_df)} questions from {question_csv}")
print(f"Questions: {questions_df['question_id'].tolist()[:5]}... (showing first 5)")

# Define the system prompt
SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about early church history.
Provide accurate, detailed, and nuanced responses based on historical scholarship.
Acknowledge uncertainty or scholarly debate where it exists.
Do not refuse historically answerable questions on the grounds of contemporary theological controversy."""

# Create output directory
output_dir = Path("../data/responses")
output_dir.mkdir(parents=True, exist_ok=True)

# Process all questions
output_file = output_dir / "openrouter.jsonl"
results = {"ok": 0, "error": 0}

with open(output_file, "w", encoding="utf-8") as f:
    for idx, row in questions_df.iterrows():
        question_id = row["question_id"]
        question_text = row["question"]
        figure = row.get("figure", "Unknown")
        
        print(f"[{idx+1}/{len(questions_df)}] Querying {question_id}...", end=" ", flush=True)
        
        try:
            response = client.chat.completions.create(
                model=os.getenv("OPENROUTER_MODEL_DEFAULT", "google/gemini-2.5-flash"),
                temperature=0.0,
                max_tokens=1024,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": question_text},
                ],
            )
            
            content = response.choices[0].message.content
            if isinstance(content, list):
                content = "".join(part.text or "" for part in content if getattr(part, "type", "") == "text")
            response_text = content
            tokens = getattr(response.usage, "total_tokens", None) or 0
            model_version = getattr(response, "model", None) or os.getenv("OPENROUTER_MODEL_DEFAULT", "google/gemini-2.5-flash")
            
            # Create record
            record = {
                "question_id": question_id,
                "figure": figure,
                "model": "openrouter",
                "model_version": model_version,
                "prompt": question_text,
                "response": response_text,
                "temperature": 0.0,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "tokens_used": tokens,
            }
            
            f.write(json.dumps(record) + "\n")
            results["ok"] += 1
            print(f"✓ ({tokens} tokens)")
            
        except Exception as e:
            results["error"] += 1
            print(f"✗ ERROR: {e}")
        
        # Rate limiting
        time.sleep(1.0)

print(f"\n=== Complete ===")
print(f"Saved {results['ok']} responses to {output_file}")

if results["error"] > 0:
    print(f"Errors: {results['error']}")